## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
#include <cstdio>
#include <vector>
#include <algorithm>

using namespace std;

// --- 全局变量与配置 ---
const int MAXN = 2005; // 根据题目需求调整大小
int N, magicA, magicB, stepL;
int currentPerm[MAXN], posLookup[MAXN];
vector<int> ops_record; // 0: magic, >0: add, <0: xor

// --- 基础操作函数 ---
void rebuild_index() {
    for (int i = 0; i < N; ++i) posLookup[currentPerm[i]] = i;
}

void do_magic() {
    ops_record.push_back(0);
    for (int i = 0; i < N; ++i) {
        if (currentPerm[i] == magicA) currentPerm[i] = magicB;
        else if (currentPerm[i] == magicB) currentPerm[i] = magicA;
    }
    rebuild_index();
}

void do_add(int offset) {
    if (offset == 0) return;
    ops_record.push_back(offset);
    for (int i = 0; i < N; ++i) currentPerm[i] = (currentPerm[i] + offset) % N;
    rebuild_index();
}

void do_xor(int mask) {
    if (mask == 0) return;
    ops_record.push_back(-mask);
    for (int i = 0; i < N; ++i) currentPerm[i] ^= mask;
    rebuild_index();
}

// --- 参数计算与置换逻辑 ---
void calc_params(int start, int end, int &pa, int &pb) {
    int diff = (end - start + 2 * N - stepL) % N;
    pa = pb = 0;
    for (int gap = N >> 1; gap >= 2 * stepL; gap >>= 1) {
        if (diff >= gap) {
            diff -= gap;
            pb += (gap >> 1);
        } else {
            pa += (gap >> 1);
        }
    }
    int offset = (start & (stepL - 1));
    pa += (N >> 1) + offset;
    pb += offset;
}

void perform_swap(int idxC, int idxD) {
    if ((idxC / stepL) % 2 == (idxD / stepL) % 2) {
        int pivot = (idxC & (stepL - 1));
        if ((idxC / stepL) % 2 == 0) pivot += stepL;
        perform_swap(idxC, pivot);
        perform_swap(idxD, pivot);
        perform_swap(idxC, pivot);
    } else {
        int tpa, tpb, tpc, tpd;
        calc_params(magicA, magicB, tpa, tpb);
        calc_params(idxC, idxD, tpc, tpd);

        do_add((tpc - idxC + N) % N);
        do_xor(tpc ^ tpa);
        do_add((magicA - tpa + N) % N);
        do_magic();
        do_add((tpa - magicA + N) % N);
        do_xor(tpc ^ tpa);
        do_add((idxC - tpc + N) % N);
    }
}

// --- 置换分解逻辑 ---
struct PState {
    int n;
    int elements[MAXN];
    vector<int> path;
};

bool decompose_p(PState &ps) {
    if (ps.n == 1) return true;
    PState left, right;
    left.n = right.n = ps.n / 2;
    for (int i = 0; i < ps.n / 2; ++i) {
        left.elements[i] = ps.elements[i * 2] / 2;
        right.elements[i] = ps.elements[i * 2 + 1] / 2;
    }
    if (!decompose_p(left) || !decompose_p(right)) return false;

    if (ps.elements[0] % 2) ps.path.push_back(ps.n == 2 ? 1 : -1);

    int b_sum = 0, c_sum = 0;
    for (int x : left.path) {
        if (x > 0) { ps.path.push_back(-1); ps.path.push_back(1); }
        else { ps.path.push_back(x * 2); b_sum ^= (-x * 2); }
    }
    if (b_sum) ps.path.push_back(-b_sum);

    for (int x : right.path) {
        if (x > 0) { ps.path.push_back(1); ps.path.push_back(-1); }
        else { ps.path.push_back(x * 2); c_sum ^= (-x * 2); }
    }

    if ((b_sum & (ps.n / 2)) != (c_sum & (ps.n / 2))) return false;
    if (b_sum % (ps.n / 2) != c_sum % (ps.n / 2)) return false;

    // 路径优化
    vector<int> opt;
    for (int x : ps.path) {
        if (!opt.empty() && x < 0 && opt.back() < 0) {
            opt.back() = -((-opt.back()) ^ (-x));
            if (opt.back() == 0) opt.pop_back();
        } else opt.push_back(x);
    }
    ps.path = opt;
    return true;
}

int main() {
    if (scanf("%d %d %d", &N, &magicA, &magicB) == EOF) return 0;
    for (int i = 0; i < N; ++i) scanf("%d", &currentPerm[i]);
    rebuild_index();

    stepL = (magicA - magicB + N) % N;
    stepL &= -stepL; // 取 lowest bit
    if (stepL == 0) stepL = N;

    if (stepL > 1) {
        PState ps;
        ps.n = stepL;
        for (int i = 0; i < N; ++i) ps.elements[i] = currentPerm[i] & (stepL - 1);
        if (!decompose_p(ps)) {
            printf("-1\n");
            return 0;
        }
        for (int cmd : ps.path) {
            if (cmd > 0) do_add(cmd);
            else do_xor(-cmd);
        }
    }

    for (int i = 0; i < stepL; ++i) {
        vector<int> group;
        for (int j = i; j < N; j += stepL) group.push_back(currentPerm[j]);
        sort(group.begin(), group.end());

        for (int j = i, idx = 0; j < N; j += stepL, ++idx) {
            if (group[idx] != j) { printf("-1\n"); return 0; }
        }
        for (int j = i; j < N; j += stepL) {
            if (currentPerm[j] != j) perform_swap(j, currentPerm[j]);
        }
    }

    printf("%zu\n", ops_record.size());
    for (int op : ops_record) {
        if (op == 0) printf("0\n");
        else if (op < 0) printf("1 %d\n", -op);
        else printf("2 %d\n", op);
    }

    return 0;
}

## B 长跑

In [ ]:
## add your code here
#include <iostream>
#include <vector>
#include <algorithm>
#include <queue>

using namespace std;

// 站点结构体，保持简单数据成员
struct Station {
    int pos;
    int fee;
};

// 排序比较函数
bool compareStations(const Station& a, const Station& b) {
    if (a.pos != b.pos) return a.pos < b.pos;
    return a.fee < b.fee;
}

// 搜索状态
struct State {
    int currP;
    int currB;
};

// 用于记忆化，记录到达某个位置时剩下的最大预算
// 如果之后以更少的预算到达同一位置，则没必要继续搜索
int max_budget_at_pos[10005]; 

void solve() {
    int totalNodes, targetDist, maxStep, initialBudget;

    // 持续读取输入，直到文件结束
    while (cin >> totalNodes >> targetDist >> maxStep >> initialBudget) {
        vector<Station> stations;
        stations.reserve(totalNodes + 1);

        for (int i = 0; i < totalNodes; ++i) {
            int p, c;
            cin >> p >> c;
            // 只有在目标范围内的站点才有意义
            if (p < targetDist) {
                stations.push_back({p, c});
            }
        }

        // 处理直接到达的情况
        if (maxStep >= targetDist) {
            cout << "Yes" << endl;
            continue;
        }

        // 加入终点并排序
        stations.push_back({targetDist, 0});
        sort(stations.begin(), stations.end(), compareStations);

        // 初始化记忆化数组（-1表示未到达）
        for (int i = 0; i <= stations.size(); ++i) max_budget_at_pos[i] = -1;

        queue<State> q;
        q.push({0, initialBudget});
        
        bool possible = false;
        while (!q.empty()) {
            State head = q.front();
            q.pop();

            if (head.currP >= targetDist) {
                possible = true;
                break;
            }

            // 尝试从当前位置跳到后面的站点
            for (int i = 0; i < stations.size(); ++i) {
                if (stations[i].pos <= head.currP) continue;
                
                int dist = stations[i].pos - head.currP;
                if (dist > maxStep) break; // 后面的更跳不过去了

                int nextBudget = head.currB - stations[i].fee;
                if (nextBudget >= 0) {
                    // 记忆化剪枝：只有当剩余预算比之前到达该点时更多时，才入队
                    if (nextBudget > max_budget_at_pos[i]) {
                        max_budget_at_pos[i] = nextBudget;
                        q.push({stations[i].pos, nextBudget});
                    }
                }
            }
            if (possible) break;
        }

        cout << (possible ? "Yes" : "No") << endl;
    }
}

int main() {
    // 提升输入输出性能
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);

    solve();

    return 0;
}

## C 最长回文

In [ ]:
## add your code here
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>

using namespace std;

typedef long long i64;
typedef unsigned long long u64;

struct RollingHash {
    static const i64 M = 999998639;
    static const i64 B = 13331;
    vector<u64> p_pow, h1, h2;

    void build(const string& s1, const string& s2, int n) {
        p_pow.assign(n + 1, 1);
        h1.assign(n + 1, 0);
        h2.assign(n + 2, 0);
        for (int i = 1; i <= n; ++i) {
            p_pow[i] = (p_pow[i - 1] * B) % M;
            h1[i] = (h1[i - 1] * B + s1[i - 1]) % M;
        }
        for (int i = n; i >= 1; --i) {
            h2[i] = (h2[i + 1] * B + s2[i - 1]) % M;
        }
    }

    u64 fetch_forward(int l, int r) {
        if (l > r) return 0xDEAD;
        i64 res = (i64)h1[r] - (i64)h1[l - 1] * p_pow[r - l + 1] % M;
        return (res % M + M) % M;
    }

    u64 fetch_backward(int l, int r) {
        if (l > r) return 0xBEEF;
        i64 res = (i64)h2[l] - (i64)h2[r + 1] * p_pow[r - l + 1] % M;
        return (res % M + M) % M;
    }
};

string transform(const string& src, vector<int>& rad) {
    string res = "@#";
    for (char c : src) {
        res += c;
        res += '#';
    }
    int m = res.size();
    rad.assign(m, 0);
    int center = 0, right_bound = 0, global_max = 0;
    for (int i = 1; i < m; ++i) {
        if (i < right_bound) rad[i] = min(right_bound - i, rad[2 * center - i]);
        else rad[i] = 1;
        while (i + rad[i] < m && i - rad[i] >= 0 && res[i + rad[i]] == res[i - rad[i]]) {
            rad[i]++;
        }
        if (i + rad[i] > right_bound) {
            center = i;
            right_bound = i + rad[i];
        }
        global_max = max(global_max, rad[i] - 1);
    }
    return res;
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int raw_len;
    string s_alpha, s_beta;
    if (!(cin >> raw_len >> s_alpha >> s_beta)) return 0;

    vector<int> rad_alpha, rad_beta;
    string t_alpha = transform(s_alpha, rad_alpha);
    string t_beta = transform(s_beta, rad_beta);

    int total_n = t_alpha.size();
    RollingHash hasher;
    hasher.build(t_alpha, t_beta, total_n);

    int max_palindrome_reach = 1;
    for (int i = 1; i <= total_n; ++i) {
        int base_rad = max(rad_alpha[i - 1], (i >= 3 ? rad_beta[i - 3] : 0));
        
        int left_edge = i - base_rad + 1;
        int right_edge = i - 2 + base_rad - 1;

        int low = 0, high = min(left_edge - 1, total_n - right_edge);
        int extra_match = 0;

        while (low <= high) {
            int mid = (low + high) >> 1;
            int L1 = max(1, left_edge - mid), R1 = left_edge - 1;
            int L2 = right_edge + 1, R2 = min(total_n, right_edge + mid);

            if (hasher.fetch_forward(L1, R1) == hasher.fetch_backward(L2, R2)) {
                extra_match = mid;
                low = mid + 1;
            } else {
                high = mid - 1;
            }
        }
        max_palindrome_reach = max(max_palindrome_reach, extra_match + base_rad - 1);
    }

    cout << max_palindrome_reach << endl;

    return 0;
}

## D 优惠券

In [ ]:
## add your code here
#include <iostream>
#include <vector>
#include <set>
#include <algorithm>

using namespace std;

const int MAX_LEN = 5e5 + 10;
char event_type[MAX_LEN];
int user_id[MAX_LEN];
int last_pos[MAX_LEN];

void solve() {
    int total_events;
    while (cin >> total_events) {
        set<int> wildcard_indices;
        int error_event = -1;
        bool found_error = false;

        // 重置状态数组（假定 ID 范围在此以内，否则改用 map）
        for (int i = 0; i < MAX_LEN; ++i) last_pos[i] = 0;

        for (int i = 1; i <= total_events; ++i) {
            string op;
            cin >> op;
            event_type[i] = op[0];

            if (event_type[i] == '?') {
                wildcard_indices.insert(i);
            } else {
                cin >> user_id[i];
            }

            if (found_error) continue;

            if (event_type[i] != '?') {
                int uid = user_id[i];
                int prev_idx = last_pos[uid];

                if (prev_idx != 0) {
                    // 状态冲突：两次进入或两次退出
                    if (event_type[prev_idx] == event_type[i]) {
                        auto it = wildcard_indices.lower_bound(prev_idx);
                        if (it == wildcard_indices.end()) {
                            error_event = i;
                            found_error = true;
                        } else {
                            wildcard_indices.erase(it);
                        }
                    }
                } else {
                    // 首次记录就是 'O' (退出)，必须消耗一个之前的 '?'
                    if (event_type[i] == 'O') {
                        auto it = wildcard_indices.lower_bound(0);
                        if (it == wildcard_indices.end()) {
                            error_event = i;
                            found_error = true;
                        } else {
                            wildcard_indices.erase(it);
                        }
                    }
                }
                last_pos[uid] = i;
            }
        }
        cout << error_event << "\n";
    }
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
    
    solve();
    
    return 0;
}

## E 任意点

In [ ]:
## add your code here
#include <iostream>
#include <vector>
#include <numeric>

using namespace std;

class DisjointSetUnion {
public:
    vector<int> parent;
    int num_components;

    DisjointSetUnion(int n) {
        parent.resize(n + 1);
        iota(parent.begin(), parent.end(), 0);
        num_components = n;
    }

    int find_set(int v) {
        if (v == parent[v]) return v;
        return parent[v] = find_set(parent[v]);
    }

    void unite_sets(int a, int b) {
        a = find_set(a);
        b = find_set(b);
        if (a != b) {
            parent[a] = b;
            num_components--;
        }
    }
};

struct Point {
    int r, c;
};

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int total_points;
    if (!(cin >> total_points)) return 0;

    vector<Point> coords(total_points);
    DisjointSetUnion dsu(total_points);

    for (int i = 0; i < total_points; ++i) {
        cin >> coords[i].r >> coords[i].c;
    }

    for (int i = 0; i < total_points; ++i) {
        for (int j = i + 1; j < total_points; ++j) {
            if (coords[i].r == coords[j].r || coords[i].c == coords[j].c) {
                dsu.unite_sets(i + 1, j + 1);
            }
        }
    }

    cout << dsu.num_components - 1 << endl;

    return 0;
}

## F 通配符匹配

In [ ]:
## add your code here
#include <cstdio>
#include <cstring>
#include <vector>
#include <iostream>
#include <algorithm>

using namespace std;

const int MAX_LEN = 100005;

// 全局位置记录与通配符标记
int match_history[MAX_LEN], history_size;
bool is_start_wild, is_end_wild;

struct Automaton {
    int next_state[MAX_LEN][26], fail_link[MAX_LEN];
    int node_count, entry_root, sub_pattern_num, total_len;
    vector<int> terminal_offsets[MAX_LEN];

    void insert_sequence(char *source, int left, int right) {
        static char buffer[MAX_LEN];
        int n = right - left;
        for (int i = 0; i < n; i++) buffer[i] = source[left + i];
        
        int curr = entry_root;
        buffer[n++] = '?'; 
        for (int i = 0; i < n; i++) {
            while (i < n && buffer[i] == '?') {
                if (i > 0 && buffer[i - 1] != '?') {
                    terminal_offsets[curr].push_back(i - 1);
                    sub_pattern_num++;
                }
                curr = entry_root;
                i++;
            }
            if (i >= n) break;
            int slot = buffer[i] - 'a';
            if (!next_state[curr][slot]) next_state[curr][slot] = ++node_count;
            curr = next_state[curr][slot];
        }
        total_len = n - 1;
    }

    void build_failure_links() {
        static int queue_buf[MAX_LEN];
        int head = 1, tail = 0;
        for (int i = 0; i < 26; i++) {
            if (next_state[entry_root][i]) queue_buf[++tail] = next_state[entry_root][i];
        }
        while (head <= tail) {
            int u = queue_buf[head++];
            for (int offset : terminal_offsets[fail_link[u]]) {
                terminal_offsets[u].push_back(offset);
            }
            for (int i = 0; i < 26; i++) {
                if (next_state[u][i]) {
                    int v = next_state[u][i];
                    fail_link[v] = next_state[fail_link[u]][i];
                    queue_buf[++tail] = v;
                } else {
                    next_state[u][i] = next_state[fail_link[u]][i];
                }
            }
        }
    }

    bool find_first_occurrence(char *text, bool is_first, bool is_last) {
        static int hit_count[MAX_LEN];
        int n = strlen(text);
        for (int i = 0; i <= n; i++) hit_count[i] = 0;

        int curr = entry_root;
        for (int i = 0; i < n; i++) {
            curr = next_state[curr][text[i] - 'a'];
            for (int offset : terminal_offsets[curr]) {
                if (i >= offset) hit_count[i - offset]++;
            }
        }

        for (int i = 0; i < n; i++) {
            if (hit_count[i] != sub_pattern_num) continue;
            
            if (history_size == 0) {
                if (!is_start_wild && is_first && i != 0) break;
                if (!is_end_wild && is_last && n - i != total_len) break;
                match_history[++history_size] = i + total_len;
                return true;
            } else {
                if (i < match_history[history_size]) continue;
                if (!is_end_wild && is_last && n - i != total_len) break;
                match_history[++history_size] = i + total_len;
                return true;
            }
        }
        return false;
    }
};

Automaton segment_ac[10];
char pattern_buf[MAX_LEN], query_buf[MAX_LEN];

int main() {
    if (scanf("%s", pattern_buf) == EOF) return 0;

    int n = strlen(pattern_buf);
    is_start_wild = (pattern_buf[0] == '*');
    is_end_wild = (pattern_buf[n - 1] == '*');

    pattern_buf[n++] = '*';
    int seg_tot = 0, last_idx = 0;
    for (int i = 0; i < n; i++) {
        if (pattern_buf[i] == '*') {
            if (i > last_idx) {
                segment_ac[seg_tot++].insert_sequence(pattern_buf, last_idx, i);
            }
            last_idx = i + 1;
        }
    }

    for (int i = 0; i < seg_tot; i++) segment_ac[i].build_failure_links();

    int queries;
    if (scanf("%d", &queries) == EOF) return 0;
    while (queries--) {
        if (scanf("%s", query_buf) == EOF) break;
        
        if (n == 1) { // 只有通配符或空串
            printf("YES\n");
            continue;
        }
        if (seg_tot == 0) {
            printf("YES\n");
            continue;
        }

        history_size = 0;
        bool possible = true;
        for (int i = 0; i < seg_tot; i++) {
            if (!segment_ac[i].find_first_occurrence(query_buf, i == 0, i == seg_tot - 1)) {
                possible = false;
                break;
            }
        }
        printf(possible ? "YES\n" : "NO\n");
    }

    return 0;
}

## G 汉诺塔

In [ ]:
## add your code here
#include <cstdio>
#include <cctype>
#include <cstring>
#include <algorithm>

using namespace std;

typedef long long i64;

// 快速输入函数，重命名并移除 register 关键字
inline char get_char() {
    static char buffer[100000], *ptr1 = buffer, *ptr2 = buffer;
    return ptr1 == ptr2 && (ptr2 = (buffer + fread(buffer, 1, 100000, stdin)), ptr1 == ptr2)
        ? EOF : *ptr1++;
}

inline i64 get_int() {
    i64 value = 0, sign = 1; 
    char ch = get_char();
    while (!isdigit(ch)) {
        if (ch == '-') sign = -1;
        ch = get_char();
    }
    while (isdigit(ch)) {
        value = (value << 1) + (value << 3) + (ch - '0');
        ch = get_char();
    }
    return value * sign;
}

inline bool isValid(char c) {
    return c >= 'A' && c <= 'C';
}

inline int getChoice() {
    char ch = get_char();
    while (!isValid(ch)) ch = get_char();
    return ch - 'A' + 1;
}

const int N_LIMIT = 35;
i64 dp_steps[4][N_LIMIT];
int target_pos[4][N_LIMIT];

int main() {
    int total_disks = get_int();

    // 初始化长度为 1 的情况
    for (int i = 1; i <= 6; ++i) {
        int from = getChoice();
        int to = getChoice();
        if (!dp_steps[from][1]) {
            dp_steps[from][1] = 1;
            target_pos[from][1] = to;
        }
    }

    // 动态规划推导规模为 i 的汉诺塔变体
    for (int i = 2; i <= total_disks; ++i) {
        for (int src = 1; src <= 3; ++src) {
            int mid = target_pos[src][i - 1];
            int alt = 6 - src - mid;

            // 基础移动代价：先将 i-1 个盘子移走，再移动底盘
            i64 current_cost = dp_steps[src][i - 1] + 1;

            if (target_pos[mid][i - 1] == alt) {
                // 情况 A：i-1 个盘子从中转柱移动到目标柱
                dp_steps[src][i] = current_cost + dp_steps[mid][i - 1];
                target_pos[src][i] = alt;
            } else {
                // 情况 B：i-1 个盘子需要先回退到源柱，再移动到中转柱
                dp_steps[src][i] = current_cost + dp_steps[mid][i - 1] + 1 + dp_steps[src][i - 1];
                target_pos[src][i] = mid;
            }
        }
    }

    printf("%lld\n", dp_steps[1][total_disks]);

    return 0;
}

## H 马步距离

In [ ]:
## add your code here
#include <cstdio>
#include <algorithm>

using namespace std;

// 坐标结构体
struct Coord {
    int r, c;
};

// 循环队列实现
struct ArrayQueue {
    int head, tail;
    Coord data[10005];

    void reset() {
        head = 0;
        tail = 0;
    }

    bool is_empty() {
        return head == tail;
    }

    void enqueue(Coord node) {
        data[tail] = node;
        tail = (tail + 1) % 10005;
    }

    Coord dequeue() {
        Coord res = data[head];
        head = (head + 1) % 10005;
        return res;
    }
} scheduler;

// 全局变量区
int dist_map[105][105];
bool visited[105][105];
int start_r, start_c;

// 马走日的偏移量
const int dr[] = {1, 1, -1, -1, 2, 2, -2, -2};
const int dc[] = {2, -2, 2, -2, 1, -1, 1, -1};

int run_bfs() {
    scheduler.reset();
    scheduler.enqueue({start_r, start_c});
    visited[start_r][start_c] = true;

    while (!visited[50][50]) {
        Coord curr = scheduler.dequeue();

        for (int i = 0; i < 8; ++i) {
            int nr = curr.r + dr[i];
            int nc = curr.c + dc[i];

            if (nr >= 0 && nr <= 100 && nc >= 0 && nc <= 100 && !visited[nr][nc]) {
                visited[nr][nc] = true;
                dist_map[nr][nc] = dist_map[curr.r][curr.c] + 1;
                scheduler.enqueue({nr, nc});
            }
        }
    }
    return dist_map[50][50];
}

int main() {
    int r1, c1, r2, c2;
    if (scanf("%d %d %d %d", &r1, &c1, &r2, &c2) == EOF) return 0;

    // 计算相对距离并归约
    start_r = abs(r1 - r2);
    start_c = abs(c1 - c2);

    int extra_steps = 0;
    while (start_r + start_c >= 50) {
        if (start_r < start_c) swap(start_r, start_c);
        
        start_r -= 4;
        if (start_r - 4 <= start_c * 2) {
            start_c -= 2;
        }
        extra_steps += 2;
    }

    // 映射到BFS网格中心
    start_r += 50;
    start_c += 50;

    int bfs_steps = run_bfs();
    printf("%d\n", extra_steps + bfs_steps);

    return 0;
}

## I 直方图最大矩形

In [ ]:
## add your code here
#include <vector>
#include <algorithm>

using namespace std;

class Solution {
public:
    int largestRectangleArea(vector<int>& heights) {
        int len = heights.size();
        if (len == 0) return 0;

        // 使用数组模拟栈，并对原数组进行边界扩充以简化逻辑
        vector<int> h;
        h.push_back(0); 
        for (int x : heights) h.push_back(x);
        h.push_back(0); 

        int n = h.size();
        int mono_stack[n];
        int top = -1;
        int max_area = 0;

        for (int i = 0; i < n; ++i) {
            // 当当前高度小于栈顶高度时，触发计算
            while (top != -1 && h[i] < h[mono_stack[top]]) {
                int mid_idx = mono_stack[top--];
                int current_h = h[mid_idx];
                
                // 左边界为新的栈顶，右边界为当前索引 i
                if (top != -1) {
                    int left_idx = mono_stack[top];
                    int width = i - left_idx - 1;
                    int area = width * current_h;
                    if (area > max_area) max_area = area;
                }
            }
            mono_stack[++top] = i;
        }

        return max_area;
    }
};

## J 消防局的设立

In [ ]:
## add your code here
#include <cstdio>
#include <iostream>
#include <vector>
#include <algorithm>

using namespace std;

const int MAXN = 10005;

struct Node {
    int id, depth;
};

// 全局变量
int n, total_ans;
int parent[MAXN];
vector<int> adj[MAXN];
Node nodes[MAXN];
bool is_covered[MAXN];

// 快速读入
inline int read() {
    int x = 0, f = 1;
    char ch = getchar();
    while (ch < '0' || ch > '9') {
        if (ch == '-') f = -1;
        ch = getchar();
    }
    while (ch >= '0' && ch <= '9') {
        x = x * 10 + ch - '0';
        ch = getchar();
    }
    return x * f;
}

// 建立树并计算深度
void dfs(int u, int p, int d) {
    parent[u] = p;
    nodes[u] = {u, d};
    for (int v : adj[u]) {
        if (v == p) continue;
        dfs(v, u, d + 1);
    }
}

// 按深度从大到小排序
bool compareDepth(Node a, Node b) {
    return a.depth > b.depth;
}

// 覆盖函数：从中心点 x 开始，覆盖半径为 2 的所有点
void cover(int u, int p, int dist) {
    is_covered[u] = true;
    if (dist <= 0) return;
    for (int v : adj[u]) {
        if (v == p) continue; // 注意：在半径覆盖中，不能只往子节点走，但此题贪心策略特殊
        cover(v, u, dist - 1);
    }
}

int main() {
    n = read();
    if (n == 0) {
        printf("0\n");
        return 0;
    }

    for (int i = 2; i <= n; i++) {
        int u = read();
        adj[i].push_back(u);
        adj[u].push_back(i);
    }

    // 以 1 为根初始化
    dfs(1, 0, 1);
    
    // 为了防止爷爷节点不存在，将根节点的父亲设为自己，父亲的父亲也设为自己
    parent[1] = 1; 
    
    sort(nodes + 1, nodes + n + 1, compareDepth);

    for (int i = 1; i <= n; i++) {
        int u = nodes[i].id;
        if (!is_covered[u]) {
            // 贪心策略：覆盖当前最深节点的爷爷节点
            int grandfather = parent[parent[u]];
            
            // 手动触发覆盖半径为 2 的逻辑
            // 在爷爷节点放置一个消防站/信号塔
            total_ans++;
            cover(grandfather, 0, 2); 
        }
    }

    printf("%d\n", total_ans);

    return 0;
}